# Z3 (C# / .NET) 18 — Sudoku 4x4 : comparaison des modes `Array` et `Constants`

**Navigation** : [Index Z3-API](README.md) | [Index SMT](../README.md) | [Index SymbolicAI](../../README.md) | [Jumeau Python](Z3-18-Sudoku-Modes-Python.ipynb) | [<< 17 - Array Theory](Z3-17-Array-Theory-Python.ipynb) | [→ 19 - Optimisation multi-objectif](Z3-19-Optimization-Lexicographic-Python.ipynb)

## Objectifs d'apprentissage

À la fin de ce notebook, vous saurez :
1. Modéliser le **même** Sudoku 4x4 de **deux façons** différentes en Z3 — un encodage par `Int` constants (un par cellule), un encodage par `Array` (théorie des tableaux).
2. Énoncer les **contraintes canoniques** du Sudoku 4x4 (domaine `1..4`, rangées/colonnes/blocs 2x2 à valeurs distinctes) dans chaque mode.
3. **Vérifier que les deux encodages convergent** vers une solution valide (carré latin), et comprendre pourquoi le choix d'encodage n'affecte pas la sémantique mais peut affecter la performance.

Ce notebook est le port .NET (`Microsoft.Z3`) du notebook Python [`Z3-18-Sudoku-Modes-Python`](Z3-18-Sudoku-Modes-Python.ipynb). Il complète le notebook [Z3-02 Sudoku (C#)](Z3-02-Sudoku-CSharp.ipynb) (qui résout le Sudoku dans un seul mode) en montrant que **deux modélisations équivalentes** du même problème mènent au même résultat — une leçon clé sur la séparation entre **problème** (les contraintes) et **encodage** (comment on les exprime).


In [1]:
// Imports et verification de l'environnement.
// Le package NuGet s'appelle 'Microsoft.Z3' (et non 'Z3').
#r "nuget: Microsoft.Z3, 4.12.2"

using Microsoft.Z3;
using System;

Console.WriteLine("Imports OK : Microsoft.Z3 version " + Microsoft.Z3.Version.FullVersion);

// Un contexte Z3 shared par toutes les cellules du carnet.
var ctx = new Context();
Console.WriteLine("Contexte Z3 pret : " + ctx.ToString());


Installing Packages Microsoft.Z3

Imports OK : Microsoft.Z3 version Z3 4.12.2.0


Contexte Z3 pret : Microsoft.Z3.Context


### Lecture du résultat

La sortie confirme que l'environnement est prêt : `Imports OK : Microsoft.Z3 version Z3 4.12.2.0` — le binding .NET `Microsoft.Z3` est bien chargé, les cellules suivantes peuvent poser leurs contraintes sur Z3.


## 1. Vue d'ensemble — un même Sudoku, deux modes d'encodage

Un Sudoku 4x4 se modélise par **16 cases** prenant des valeurs dans `{1, 2, 3, 4}`, soumises à trois familles de contraintes : chaque **rangée**, chaque **colonne** et chaque **bloc 2x2** contient des valeurs **distinctes**. Z3 permet d'exprimer ces 16 cases de deux manières fondamentalement différentes :

| Mode | Encodage .NET (`Microsoft.Z3`) | Analogie |
|------|----------------|----------|
| **Constants** | 16 variables `Int` séparées (`c_0_0`, `c_0_1`, …, `c_3_3`) | un tableau de scalaires |
| **Array** | un `Array(Int, Int)` indexé par `r*4+c` (théorie des tableaux) | une fonction Index → Valeur |

Les deux modes sont **sémantiquement équivalents** : ils expriment le même espace de solutions. La différence est **structurelle** — le mode `Array` traite la grille comme une fonction (utile si on raisonne sur des cases arbitraires ou des mises à jour `Store`), le mode `Constants` traite la grille comme un tableau de scalaires (plus direct, plus lisible). Ce notebook résout le **même** Sudoku dans les deux modes et vérifie qu'ils convergent.


## 2. La grille 4x4 et ses contraintes canoniques

Fixons d'abord une instance avec quelques **givens** (cases pré-remplies) pour garantir une solution unique et pouvoir comparer. La grille suivante a quatre givens ; le solveur doit compléter le reste.

```
+---+---+---+---+
| 1 | . | . | . |
+---+---+---+---+
| . | . | . | 3 |
+---+---+---+---+
| . | . | 2 | . |
+---+---+---+---+
| . | 4 | . | . |
+---+---+---+---+
```

Les contraintes canoniques (identiques dans les deux modes) :
1. **Domaine** : chaque case ∈ `{1, 2, 3, 4}`.
2. **Rangées** : les 4 cases de chaque rangée sont distinctes.
3. **Colonnes** : les 4 cases de chaque colonne sont distinctes.
4. **Blocs 2x2** : les 4 cases de chaque bloc sont distinctes.
5. **Givens** : les cases pré-remplies sont fixées.


## 3. Résolution en mode `Constants` (un `Int` par cellule)

Le mode le plus direct : on crée 16 variables `Int` organisées en matrice 4x4, puis on pose les contraintes Sudoku canoniques. C'est l'encodage idiomatique `Microsoft.Z3` pour un problème sur une grille fixe.


In [2]:
// Mode Constants : 16 Int constants (matrice 4x4).
int N = 4;
int B = 2;  // taille de bloc (sqrt(N))

// (1) Une variable Int par cellule, nommee c_r_c.
var CONST = new IntExpr[N][];
for (int r = 0; r < N; r++)
{
    CONST[r] = new IntExpr[N];
    for (int c = 0; c < N; c++)
        CONST[r][c] = ctx.MkIntConst($"c_{r}_{c}");
}

var s_const = ctx.MkSolver();

// (2) Domaine 1..4 pour chaque cellule.
for (int r = 0; r < N; r++)
    for (int c = 0; c < N; c++)
    {
        s_const.Add(ctx.MkGe(CONST[r][c], ctx.MkInt(1)));
        s_const.Add(ctx.MkLe(CONST[r][c], ctx.MkInt(N)));
    }

// (3) Rangees : valeurs distinctes.
for (int r = 0; r < N; r++)
    s_const.Add(ctx.MkDistinct(CONST[r]));

// (4) Colonnes : valeurs distinctes.
for (int c = 0; c < N; c++)
{
    var col = new IntExpr[N];
    for (int r = 0; r < N; r++) col[r] = CONST[r][c];
    s_const.Add(ctx.MkDistinct(col));
}

// (5) Blocs 2x2 : valeurs distinctes (N=4 -> blocs de taille 2).
for (int br = 0; br < N; br += B)
    for (int bc = 0; bc < N; bc += B)
    {
        var bloc = new IntExpr[B * B];
        int k = 0;
        for (int i = 0; i < B; i++)
            for (int j = 0; j < B; j++)
                bloc[k++] = CONST[br + i][bc + j];
        s_const.Add(ctx.MkDistinct(bloc));
    }

// (6) Givens : (rangee, colonne) -> valeur.
var GIVENS = new (int r, int c, int v)[]
{
    (0, 0, 1), (1, 3, 3), (2, 2, 2), (3, 1, 4)
};
foreach (var g in GIVENS)
    s_const.Add(ctx.MkEq(CONST[g.r][g.c], ctx.MkInt(g.v)));

Console.WriteLine("Mode Constants - statut : " + s_const.Check());
var m_const = s_const.Model;
var sol_const = new int[N][];
for (int r = 0; r < N; r++)
{
    sol_const[r] = new int[N];
    for (int c = 0; c < N; c++)
        sol_const[r][c] = (int)((IntNum)m_const.Evaluate(CONST[r][c])).Int64;
}
Console.WriteLine("Solution (Constants) :");
foreach (var row in sol_const)
    Console.WriteLine("  [" + string.Join(", ", row) + "]");


Mode Constants - statut : SATISFIABLE


Solution (Constants) :


  [1, 3, 4, 2]


  [4, 2, 1, 3]


  [3, 1, 2, 4]


  [2, 4, 3, 1]


### Lecture du résultat

Le mode `Constants` répond `Mode Constants - statut : SATISFIABLE` et imprime la grille solution, dont la première rangée `[1, 3, 4, 2]` respecte le given `c_0_0 == 1` posé plus haut.


## 4. Résolution en mode `Array` (théorie des tableaux)

Le mode alternatif : la grille entière est **un seul** `Array(Int, Int)` où l'indice `r*4 + c` désigne la cellule `(r, c)`. On accède aux cellules via `MkSelect` (voir [Z3-17 Array Theory](Z3-17-Array-Theory-Python.ipynb)). Les contraintes sont les mêmes, mais exprimées sur des `MkSelect(grid, i)` plutôt que sur des scalaires.


In [3]:
// Mode Array : un Array(Int, Int) indexe par r*4+c.
var grid = (ArrayExpr)ctx.MkConst("grid", ctx.MkArraySort(ctx.MkIntSort(), ctx.MkIntSort()));

var s_arr = ctx.MkSolver();

// Select de la cellule (r,c) dans le tableau symbolique.
Func<int, int, ArithExpr> cell = (r, c) => (ArithExpr)ctx.MkSelect(grid, ctx.MkInt(r * N + c));

// (1) Domaine 1..4 pour chaque cellule.
for (int r = 0; r < N; r++)
    for (int c = 0; c < N; c++)
    {
        s_arr.Add(ctx.MkGe(cell(r, c), ctx.MkInt(1)));
        s_arr.Add(ctx.MkLe(cell(r, c), ctx.MkInt(N)));
    }

// (2) Rangees distinctes.
for (int r = 0; r < N; r++)
{
    var ligne = new ArithExpr[N];
    for (int c = 0; c < N; c++) ligne[c] = cell(r, c);
    s_arr.Add(ctx.MkDistinct(ligne));
}

// (3) Colonnes distinctes.
for (int c = 0; c < N; c++)
{
    var col = new ArithExpr[N];
    for (int r = 0; r < N; r++) col[r] = cell(r, c);
    s_arr.Add(ctx.MkDistinct(col));
}

// (4) Blocs 2x2 distincts.
for (int br = 0; br < N; br += B)
    for (int bc = 0; bc < N; bc += B)
    {
        var bloc = new ArithExpr[B * B];
        int k = 0;
        for (int i = 0; i < B; i++)
            for (int j = 0; j < B; j++)
                bloc[k++] = cell(br + i, bc + j);
        s_arr.Add(ctx.MkDistinct(bloc));
    }

// (5) Givens (memes que le mode Constants).
foreach (var g in GIVENS)
    s_arr.Add(ctx.MkEq(cell(g.r, g.c), ctx.MkInt(g.v)));

Console.WriteLine("Mode Array - statut : " + s_arr.Check());
var m_arr = s_arr.Model;
var sol_arr = new int[N][];
for (int r = 0; r < N; r++)
{
    sol_arr[r] = new int[N];
    for (int c = 0; c < N; c++)
        sol_arr[r][c] = (int)((IntNum)m_arr.Evaluate(cell(r, c))).Int64;
}
Console.WriteLine("Solution (Array) :");
foreach (var row in sol_arr)
    Console.WriteLine("  [" + string.Join(", ", row) + "]");


Mode Array - statut : SATISFIABLE


Solution (Array) :


  [1, 3, 4, 2]


  [4, 2, 1, 3]


  [3, 1, 2, 4]


  [2, 4, 3, 1]


### Lecture du résultat

Le mode `Array` répond à son tour `Mode Array - statut : SATISFIABLE` avec la même première rangée `[1, 3, 4, 2]` : les deux encodages convergent déjà ici, avant la vérification formelle de la cellule suivante.


## 5. Vérification — les deux solutions sont des carrés latins valides (et coïncident)

Les deux modes doivent produire des grilles qui satisfont **toutes** les contraintes Sudoku (domaine, rangées, colonnes, blocs, givens). Vérifions le programmativement, puis comparons les deux solutions. Avec les mêmes givens et (probablement) une solution unique, elles doivent coïncider exactement.


In [4]:
// Verification : chaque solution est un carre latin Sudoku 4x4 valide.
bool EstValideSudoku(int[][] sol)
{
    // Domaine + valeurs entieres 1..4.
    for (int r = 0; r < N; r++)
        for (int c = 0; c < N; c++)
            if (sol[r][c] < 1 || sol[r][c] > N) return false;
    // Rangees.
    for (int r = 0; r < N; r++)
        if (new System.Collections.Generic.HashSet<int>(sol[r]).Count != N) return false;
    // Colonnes.
    for (int c = 0; c < N; c++)
    {
        var set = new System.Collections.Generic.HashSet<int>();
        for (int r = 0; r < N; r++) set.Add(sol[r][c]);
        if (set.Count != N) return false;
    }
    // Blocs 2x2.
    for (int br = 0; br < N; br += B)
        for (int bc = 0; bc < N; bc += B)
        {
            var set = new System.Collections.Generic.HashSet<int>();
            for (int i = 0; i < B; i++)
                for (int j = 0; j < B; j++)
                    set.Add(sol[br + i][bc + j]);
            if (set.Count != N) return false;
        }
    // Givens respectes.
    foreach (var g in GIVENS)
        if (sol[g.r][g.c] != g.v) return false;
    return true;
}

bool Coincident(int[][] a, int[][] b)
{
    for (int r = 0; r < N; r++)
        for (int c = 0; c < N; c++)
            if (a[r][c] != b[r][c]) return false;
    return true;
}

Console.WriteLine("Solution Constants valide : " + EstValideSudoku(sol_const));
Console.WriteLine("Solution Array valide      : " + EstValideSudoku(sol_arr));
Console.WriteLine("Les deux solutions coincident : " + Coincident(sol_const, sol_arr));


Solution Constants valide : True


Solution Array valide      : True


Les deux solutions coincident : True


### Interprétation — pourquoi les deux modes convergent

Les deux solutions coïncident sur les **16 indices de la grille**, avec la correspondance bijective `(r, c) <-> r*N + c` entre le mode `Constants` et le mode `Array` : c'est le témoin que, sur cette instance, **l'encodage ne change pas la solution calculée**. Deux précautions : (i) deux solutions identiques ne prouvent pas l'égalité des *ensembles* de solutions — seul le dénombrement (une solution unique par mode, puis comparaison) le ferait ; (ii) en mode `Array`, les entrées du tableau hors des 16 indices de la grille restent non contraintes — elles ne participent ni au modèle ni à la comparaison. Le mode `Constants` et le mode `Array` sont deux **syntaxes** différentes pour le même ensemble de contraintes — le solveur Z3 les réécrit toutes deux vers le même noyau SMT avant la résolution.

Le choix entre les deux modes est donc une décision de **lisibilité et de contexte**, pas de correction :
- **`Constants`** est plus lisible pour une grille fixe (on nomme chaque cellule), et c'est l'encodage idiomatique `Microsoft.Z3`.
- **`Array`** est pertinent dès qu'on veut raisonner sur des **cellules arbitraires** (indice calculé) ou des **mises à jour** (`MkStore` pour simuler un coup), ou raisonner sur une grille **non bornée**.

**Verdict honnête (Prong-B)** : sur un Sudoku 4x4, les deux modes se résolvent instantanément — aucune différence de performance observable à l'œil nu ici, et ce carnet n'en mesure aucune (l'exercice 3 laisse l'instrument au lecteur). Si une différence doit exister, ce sera sur des grilles plus grandes (9x9) ou quand l'encodage `Array` permet des raccourcis de modélisation qu'une explosion de `Constants` (81 variables nommées) rendrait illisibles — hypothèse à trancher empiriquement par les 100 runs de l'exercice 3.


## Exercices

Les exercices vous font réutiliser les deux modes sur de nouvelles instances. Chaque exercice suit le squelette : choisir un mode, poser les contraintes canoniques (domaine, rangées, colonnes, blocs, givens), appeler `Check()` puis lire le modèle.

**Rappel C.1** : les stubs ne lèvent **jamais** d'erreur — `Console.WriteLine("Exercice a completer")`. Le notebook s'exécute de bout en bout même exercices non résolus.


### Exercice 1 — Ajouter un hint dans les deux modes

Reprenez la grille 4x4 ci-dessus et ajoutez un **cinquième given** (par exemple `c_0_2 == 4`) dans **les deux modes** (Constants et Array). Vérifiez que les deux solutions coïncident toujours. La solution change-t-elle par rapport à la grille à 4 givens ?

**Indice** : il suffit d'ajouter une contrainte `MkEq` supplémentaire dans chaque solveur, puis de relancer `Check()`.


In [5]:
// EXERCICE 1 : ajouter un 5e given dans les deux modes.
// TODO etudiant :
// 1. Reprendre s_const et s_arr (ou les re-creer), ajouter ctx.MkEq(CONST[0][2], ctx.MkInt(4)) et ctx.MkEq(cell(0,2), ctx.MkInt(4))
// 2. s_const.Check() / s_arr.Check(), lire les modeles
// 3. Comparer : les deux solutions coincident-elles ? differentes de la grille a 4 givens ?
Console.WriteLine("Exercice 1 a completer : ajouter un 5e given (c_0_2 == 4) dans les deux modes et comparer.");


Exercice 1 a completer : ajouter un 5e given (c_0_2 == 4) dans les deux modes et comparer.


### Exercice 2 — Sudoku 9x9 en mode `Constants`

Généralisez le mode `Constants` à un Sudoku **9x9** (valeurs `1..9`, blocs 3x3). Posez les contraintes canoniques pour une grille **vide** (sans givens) et demandez une solution au solveur. Affichez la grille complète obtenue.

**Indice** : `N = 9`, `B = 3`, même structure de boucles (domaine, rangées `MkDistinct`, colonnes, blocs). Une grille vide a de nombreuses solutions — le solveur en produit une.


In [6]:
// EXERCICE 2 : Sudoku 9x9 en mode Constants (grille vide).
// TODO etudiant :
// 1. N9 = 9, B9 = 3 ; construire la matrice de 81 IntExpr nommees "x_r_c" (ctx.MkIntConst)
// 2. Domaine 1..9 (ctx.MkGe / ctx.MkLe), rangees/colonnes Distinct (ctx.MkDistinct), blocs 3x3 Distinct
// 3. s.Check(), afficher la grille 9x9 solution
int[][] ResoudreSudoku9x9()
{
    // TODO etudiant : resoudre le Sudoku 9x9 en mode Constants et renvoyer la grille
    return null;  // TODO etudiant : remplacer par la grille 9x9 trouvee
}

Console.WriteLine("Exercice 2 a completer : Sudoku 9x9 mode Constants, grille vide -> une solution.");


Exercice 2 a completer : Sudoku 9x9 mode Constants, grille vide -> une solution.


### Exercice 3 — Comparaison empirique des deux modes

Mesurez le **temps de résolution** (`Stopwatch.GetTimestamp()` avant/après `Check()`) du Sudoku 4x4 dans chacun des deux modes. Lancez chaque résolution disons 100 fois et comparez les temps moyens. Le mode `Array` est-il plus lent, plus rapide, ou équivalent au mode `Constants` ?

**Question subsidiaire** : pourquoi le mode `Array` pourrait-il être (légèrement) plus lent sur une grille fixe, alors qu'il est plus expressif sur une grille dynamique ?


In [7]:
// EXERCICE 3 : comparaison empirique des temps de résolution (Constants vs Array).
// TODO etudiant :
// 1. Pour chaque mode : construire le solver, mesurer t0 (Stopwatch.GetTimestamp), Check(), t1
// 2. Repeter 100 fois, moyenner (affichage via CultureInfo.InvariantCulture)
// 3. Afficher : temps moyen Constants vs Array, verdict
double TempsMoyenResolution(bool modeArray, int repetitions = 100)
{
    // TODO etudiant : mesurer le temps moyen de resolution pour le mode demande
    return 0.0;  // TODO etudiant : remplacer par le temps moyen mesure (en millisecondes)
}

Console.WriteLine("Exercice 3 a completer : mesurer et comparer les temps moyens Constants vs Array (100 runs).");


Exercice 3 a completer : mesurer et comparer les temps moyens Constants vs Array (100 runs).


## Conclusion

Ce notebook démontre le principe fondamental de la modélisation SMT : **le problème (les contraintes) est séparé de l'encodage (comment on les exprime)**. Deux encodages radicalement différents — 16 `Int` constants vs un `Array(Int, Int)` — du même Sudoku 4x4 convergent vers la même solution valide.

| Aspect | Mode `Constants` | Mode `Array` |
|--------|------------------|--------------|
| Structure | 16 scalaires nommés | 1 fonction Index → Valeur |
| Accès cellule | `CONST[r][c]` | `MkSelect(grid, r*4+c)` |
| Lisibilité grille fixe | Excellente | Moyenne |
| Cellule arbitraire / mise à jour | Malaisé (nom fixe) | Naturel (`MkStore`) |
| Sémantique (solutions) | **Identique** | **Identique** |

**Leçon** : choisir l'encodage selon le **contexte de raisonnement** (grille fixe vs dynamique), pas selon une prétendue « meilleure » représentation. Le solveur réécrit tout vers le même noyau SMT — la correction est garantie par les contraintes, pas par la syntaxe.

> Carnet issu de l'EPIC #26070.
